<a href="https://colab.research.google.com/github/brunacorreiade/APIs/blob/master/analise_ingressantes_inep_2015.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# ============================================================
# ANÁLISE DOS INGRESSANTES — CENSO DA EDUCAÇÃO SUPERIOR 2015
# Fonte: Inep
# ============================================================

import pandas as pd
import zipfile
import shutil

from pathlib import Path
from google.colab import files
from IPython.display import display


# ------------------------------------------------------------
# 1. ENVIAR O ZIP DE 2015
# ------------------------------------------------------------

print("Selecione o ZIP dos microdados de 2015.")

arquivos_enviados = files.upload()

arquivos_zip = [
    nome for nome in arquivos_enviados
    if nome.lower().endswith(".zip")
]

if not arquivos_zip:
    raise FileNotFoundError("Nenhum arquivo ZIP foi enviado.")

nome_zip = next(
    (nome for nome in arquivos_zip if "2015" in nome),
    arquivos_zip[0]
)

caminho_zip = Path("/content") / nome_zip

print(f"\nArquivo escolhido: {caminho_zip.name}")


# ------------------------------------------------------------
# 2. LOCALIZAR E EXTRAIR O CSV DE CURSOS
# ------------------------------------------------------------

pasta_extracao = Path("/content/inep_2015")
pasta_extracao.mkdir(exist_ok=True)

with zipfile.ZipFile(caminho_zip, "r") as arquivo_zip:
    arquivos_internos = arquivo_zip.namelist()

    arquivo_cursos = next(
        (
            arquivo for arquivo in arquivos_internos
            if arquivo.upper().endswith(
                "MICRODADOS_CADASTRO_CURSOS_2015.CSV"
            )
        ),
        None
    )

    if arquivo_cursos is None:
        raise FileNotFoundError(
            "O CSV de cursos de 2015 não foi encontrado dentro do ZIP."
        )

    arquivo_zip.extract(arquivo_cursos, pasta_extracao)

caminho_csv = pasta_extracao / arquivo_cursos

print(f"CSV localizado: {caminho_csv.name}")


# ------------------------------------------------------------
# 3. DEFINIR AS VARIÁVEIS
# ------------------------------------------------------------

colunas_idade = [
    "QT_ING_0_17",
    "QT_ING_18_24",
    "QT_ING_25_29",
    "QT_ING_30_34",
    "QT_ING_35_39",
    "QT_ING_40_49",
    "QT_ING_50_59",
    "QT_ING_60_MAIS"
]

colunas_30_mais = [
    "QT_ING_30_34",
    "QT_ING_35_39",
    "QT_ING_40_49",
    "QT_ING_50_59",
    "QT_ING_60_MAIS"
]

colunas_contagem = [
    "QT_ING",
    "QT_ING_DIURNO",
    "QT_ING_NOTURNO"
] + colunas_idade

colunas_utilizadas = [
    "NU_ANO_CENSO",
    "TP_DIMENSAO",
    "TP_CATEGORIA_ADMINISTRATIVA",
    "TP_REDE",
    "CO_IES",
    "CO_CURSO",
    "NO_CURSO",
    "CO_CINE_ROTULO",
    "NO_CINE_ROTULO",
    "TP_MODALIDADE_ENSINO"
] + colunas_contagem


# ------------------------------------------------------------
# 4. ABRIR A BASE
# ------------------------------------------------------------

dados = pd.read_csv(
    caminho_csv,
    sep=";",
    encoding="latin1",
    usecols=colunas_utilizadas,
    na_values=["."],
    low_memory=False,
    dtype={
        "CO_IES": "string",
        "CO_CURSO": "string",
        "CO_CINE_ROTULO": "string"
    }
)

print(f"\nLinhas: {len(dados):,}")
print(f"Colunas utilizadas: {len(dados.columns)}")


# ------------------------------------------------------------
# 5. LIMPAR AS VARIÁVEIS
# ------------------------------------------------------------

for coluna in colunas_contagem:
    dados[coluna] = (
        pd.to_numeric(dados[coluna], errors="coerce")
        .fillna(0)
        .astype("int64")
    )

dados["CO_CINE_ROTULO"] = (
    dados["CO_CINE_ROTULO"]
    .str.replace('"', "", regex=False)
    .str.strip()
)

mapa_modalidade = {
    1: "Presencial",
    2: "EaD"
}

mapa_rede = {
    1: "Pública",
    2: "Privada"
}

mapa_categoria = {
    1: "Pública federal",
    2: "Pública estadual",
    3: "Pública municipal",
    4: "Privada com fins lucrativos",
    5: "Privada sem fins lucrativos",
    6: "Privada particular",
    7: "Especial",
    8: "Privada comunitária",
    9: "Privada confessional"
}

dados["modalidade"] = (
    dados["TP_MODALIDADE_ENSINO"].map(mapa_modalidade)
)

dados["rede"] = dados["TP_REDE"].map(mapa_rede)

dados["categoria_administrativa"] = (
    dados["TP_CATEGORIA_ADMINISTRATIVA"].map(mapa_categoria)
)

dados["ingressantes_30_mais"] = (
    dados[colunas_30_mais].sum(axis=1)
)


# ------------------------------------------------------------
# 6. CHECAR OS TOTAIS
# ------------------------------------------------------------

total_ingressantes = int(dados["QT_ING"].sum())
total_faixas = int(dados[colunas_idade].sum().sum())
total_30_mais = int(dados["ingressantes_30_mais"].sum())

linhas_inconsistentes = int(
    (
        dados[colunas_idade].sum(axis=1)
        != dados["QT_ING"]
    ).sum()
)

checagens = pd.DataFrame([
    {
        "checagem": "Total de ingressantes",
        "valor": total_ingressantes,
        "resultado_esperado": 2_922_400,
        "status": (
            "OK"
            if total_ingressantes == 2_922_400
            else "REVISAR"
        )
    },
    {
        "checagem": "Soma das faixas etárias",
        "valor": total_faixas,
        "resultado_esperado": total_ingressantes,
        "status": (
            "OK"
            if total_faixas == total_ingressantes
            else "REVISAR"
        )
    },
    {
        "checagem": "Linhas em que idade não soma QT_ING",
        "valor": linhas_inconsistentes,
        "resultado_esperado": 0,
        "status": (
            "OK"
            if linhas_inconsistentes == 0
            else "REVISAR"
        )
    }
])

print("\nCHECAGENS")
display(checagens)

assert total_ingressantes == 2_922_400, (
    "O total não corresponde ao resultado oficial de 2015."
)

assert total_faixas == total_ingressantes, (
    "As faixas etárias não somam o total de ingressantes."
)


# ------------------------------------------------------------
# 7. TOTAL POR FAIXA ETÁRIA
# ------------------------------------------------------------

nomes_faixas = {
    "QT_ING_0_17": "Até 17",
    "QT_ING_18_24": "18 a 24",
    "QT_ING_25_29": "25 a 29",
    "QT_ING_30_34": "30 a 34",
    "QT_ING_35_39": "35 a 39",
    "QT_ING_40_49": "40 a 49",
    "QT_ING_50_59": "50 a 59",
    "QT_ING_60_MAIS": "60 ou mais"
}

ordem_faixas = list(nomes_faixas.values())

idade_brasil = (
    dados[colunas_idade]
    .sum()
    .rename_axis("variavel")
    .reset_index(name="ingressantes")
)

idade_brasil["faixa_etaria"] = (
    idade_brasil["variavel"].map(nomes_faixas)
)

idade_brasil["percentual_total"] = (
    idade_brasil["ingressantes"] / total_ingressantes
)

idade_brasil = idade_brasil[
    ["faixa_etaria", "ingressantes", "percentual_total"]
]

print("\nINGRESSANTES POR FAIXA ETÁRIA")
display(
    idade_brasil.style.format({
        "ingressantes": "{:,.0f}",
        "percentual_total": "{:.2%}"
    })
)


# ------------------------------------------------------------
# 8. RESUMO DOS INGRESSANTES COM 30 ANOS OU MAIS
# ------------------------------------------------------------

resumo_geral = pd.DataFrame([
    {
        "ano": 2015,
        "total_ingressantes": total_ingressantes,
        "ingressantes_30_mais": total_30_mais,
        "percentual_30_mais": (
            total_30_mais / total_ingressantes
        )
    }
])

print("\nRESUMO GERAL")
display(
    resumo_geral.style.format({
        "total_ingressantes": "{:,.0f}",
        "ingressantes_30_mais": "{:,.0f}",
        "percentual_30_mais": "{:.2%}"
    })
)


# ------------------------------------------------------------
# 9. 30+ POR MODALIDADE E REDE
# ------------------------------------------------------------

resumo_30_modalidade_rede = (
    dados
    .groupby(["modalidade", "rede"], as_index=False)
    .agg(
        total_ingressantes=("QT_ING", "sum"),
        ingressantes_30_mais=("ingressantes_30_mais", "sum")
    )
)

resumo_30_modalidade_rede["percentual_30_mais"] = (
    resumo_30_modalidade_rede["ingressantes_30_mais"]
    / resumo_30_modalidade_rede["total_ingressantes"]
)

resumo_30_modalidade_rede["participacao_total_30_mais"] = (
    resumo_30_modalidade_rede["ingressantes_30_mais"]
    / resumo_30_modalidade_rede[
        "ingressantes_30_mais"
    ].sum()
)

print("\n30+ POR MODALIDADE E REDE")
display(
    resumo_30_modalidade_rede.style.format({
        "total_ingressantes": "{:,.0f}",
        "ingressantes_30_mais": "{:,.0f}",
        "percentual_30_mais": "{:.2%}",
        "participacao_total_30_mais": "{:.2%}"
    })
)


# ------------------------------------------------------------
# 10. 30+ POR CATEGORIA ADMINISTRATIVA
# ------------------------------------------------------------

resumo_categoria = (
    dados
    .groupby("categoria_administrativa", as_index=False)
    .agg(
        total_ingressantes=("QT_ING", "sum"),
        ingressantes_30_mais=("ingressantes_30_mais", "sum")
    )
)

resumo_categoria["percentual_30_mais"] = (
    resumo_categoria["ingressantes_30_mais"]
    / resumo_categoria["total_ingressantes"]
)

resumo_categoria["participacao_total_30_mais"] = (
    resumo_categoria["ingressantes_30_mais"]
    / total_30_mais
)

print("\n30+ POR CATEGORIA ADMINISTRATIVA")
display(
    resumo_categoria.style.format({
        "total_ingressantes": "{:,.0f}",
        "ingressantes_30_mais": "{:,.0f}",
        "percentual_30_mais": "{:.2%}",
        "participacao_total_30_mais": "{:.2%}"
    })
)


# ------------------------------------------------------------
# 11. FAIXA ETÁRIA × MODALIDADE × REDE
# ------------------------------------------------------------

faixa_modalidade_rede = dados.melt(
    id_vars=["modalidade", "rede"],
    value_vars=colunas_idade,
    var_name="variavel_idade",
    value_name="ingressantes"
)

faixa_modalidade_rede["faixa_etaria"] = (
    faixa_modalidade_rede["variavel_idade"].map(nomes_faixas)
)

faixa_modalidade_rede = (
    faixa_modalidade_rede
    .groupby(
        ["faixa_etaria", "modalidade", "rede"],
        as_index=False
    )["ingressantes"]
    .sum()
)

faixa_modalidade_rede["percentual_no_grupo"] = (
    faixa_modalidade_rede
    .groupby(["modalidade", "rede"])["ingressantes"]
    .transform(lambda valores: valores / valores.sum())
)

faixa_modalidade_rede["faixa_etaria"] = pd.Categorical(
    faixa_modalidade_rede["faixa_etaria"],
    categories=ordem_faixas,
    ordered=True
)

faixa_modalidade_rede = (
    faixa_modalidade_rede
    .sort_values(["faixa_etaria", "modalidade", "rede"])
)

print("\nFAIXA ETÁRIA × MODALIDADE × REDE")
display(
    faixa_modalidade_rede.style.format({
        "ingressantes": "{:,.0f}",
        "percentual_no_grupo": "{:.2%}"
    })
)


# ------------------------------------------------------------
# 12. PEDAGOGIA E SERVIÇO SOCIAL
# ------------------------------------------------------------

cursos_pauta = {
    "0113P01": "Pedagogia",
    "0923S01": "Serviço Social"
}

dados["curso_pauta"] = (
    dados["CO_CINE_ROTULO"].map(cursos_pauta)
)

dados_cursos = dados[
    dados["curso_pauta"].notna()
].copy()

resumo_cursos = (
    dados_cursos
    .groupby(
        ["curso_pauta", "modalidade", "rede"],
        as_index=False
    )
    .agg(
        total_ingressantes=("QT_ING", "sum"),
        ingressantes_30_mais=("ingressantes_30_mais", "sum")
    )
)

resumo_cursos["percentual_30_mais"] = (
    resumo_cursos["ingressantes_30_mais"]
    / resumo_cursos["total_ingressantes"]
)

print("\nPEDAGOGIA E SERVIÇO SOCIAL — 30+")
display(
    resumo_cursos.style.format({
        "total_ingressantes": "{:,.0f}",
        "ingressantes_30_mais": "{:,.0f}",
        "percentual_30_mais": "{:.2%}"
    })
)

faixas_cursos = dados_cursos.melt(
    id_vars=["curso_pauta", "modalidade", "rede"],
    value_vars=colunas_idade,
    var_name="variavel_idade",
    value_name="ingressantes"
)

faixas_cursos["faixa_etaria"] = (
    faixas_cursos["variavel_idade"].map(nomes_faixas)
)

faixas_cursos = (
    faixas_cursos
    .groupby(
        [
            "curso_pauta",
            "faixa_etaria",
            "modalidade",
            "rede"
        ],
        as_index=False
    )["ingressantes"]
    .sum()
)

faixas_cursos["faixa_etaria"] = pd.Categorical(
    faixas_cursos["faixa_etaria"],
    categories=ordem_faixas,
    ordered=True
)

faixas_cursos = faixas_cursos.sort_values(
    ["curso_pauta", "faixa_etaria", "modalidade", "rede"]
)


# ------------------------------------------------------------
# 13. TURNO DOS CURSOS PRESENCIAIS
# ------------------------------------------------------------
# ATENÇÃO:
# A base permite calcular o total por turno, mas não permite
# cruzar faixa etária e turno.

presencial = dados[
    dados["modalidade"] == "Presencial"
]

turnos_presenciais = pd.DataFrame({
    "turno": ["Diurno", "Noturno"],
    "ingressantes": [
        presencial["QT_ING_DIURNO"].sum(),
        presencial["QT_ING_NOTURNO"].sum()
    ]
})

turnos_presenciais["percentual"] = (
    turnos_presenciais["ingressantes"]
    / turnos_presenciais["ingressantes"].sum()
)

print("\nTURNOS — CURSOS PRESENCIAIS")
display(
    turnos_presenciais.style.format({
        "ingressantes": "{:,.0f}",
        "percentual": "{:.2%}"
    })
)


# ------------------------------------------------------------
# 14. CRIAR PASTA DO PROJETO
# ------------------------------------------------------------

pasta_projeto = Path("/content/analise_inep_2015")
pasta_resultados = pasta_projeto / "resultados"

pasta_projeto.mkdir(exist_ok=True)
pasta_resultados.mkdir(exist_ok=True)


# ------------------------------------------------------------
# 15. EXPORTAR OS CSVs PARA O GITHUB
# ------------------------------------------------------------

idade_brasil.to_csv(
    pasta_resultados / "idade_brasil_2015.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_geral.to_csv(
    pasta_resultados / "resumo_geral_2015.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_30_modalidade_rede.to_csv(
    pasta_resultados / "30_mais_modalidade_rede_2015.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_categoria.to_csv(
    pasta_resultados / "categoria_administrativa_2015.csv",
    index=False,
    encoding="utf-8-sig"
)

faixa_modalidade_rede.to_csv(
    pasta_resultados / "faixa_modalidade_rede_2015.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_cursos.to_csv(
    pasta_resultados / "pedagogia_servico_social_2015.csv",
    index=False,
    encoding="utf-8-sig"
)

faixas_cursos.to_csv(
    pasta_resultados / "cursos_por_faixa_2015.csv",
    index=False,
    encoding="utf-8-sig"
)

turnos_presenciais.to_csv(
    pasta_resultados / "turnos_presenciais_2015.csv",
    index=False,
    encoding="utf-8-sig"
)

checagens.to_csv(
    pasta_resultados / "checagens_2015.csv",
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 16. EXPORTAR UMA PLANILHA EXCEL
# ------------------------------------------------------------

caminho_excel = pasta_projeto / "resultados_inep_2015.xlsx"

with pd.ExcelWriter(
    caminho_excel,
    engine="openpyxl"
) as writer:

    checagens.to_excel(
        writer,
        sheet_name="Checagens",
        index=False
    )

    resumo_geral.to_excel(
        writer,
        sheet_name="Resumo geral",
        index=False
    )

    idade_brasil.to_excel(
        writer,
        sheet_name="Faixas etárias",
        index=False
    )

    resumo_30_modalidade_rede.to_excel(
        writer,
        sheet_name="30 mais modalidade rede",
        index=False
    )

    resumo_categoria.to_excel(
        writer,
        sheet_name="Categoria administrativa",
        index=False
    )

    faixa_modalidade_rede.to_excel(
        writer,
        sheet_name="Faixa modalidade rede",
        index=False
    )

    resumo_cursos.to_excel(
        writer,
        sheet_name="Cursos 30 mais",
        index=False
    )

    faixas_cursos.to_excel(
        writer,
        sheet_name="Cursos por faixa",
        index=False
    )

    turnos_presenciais.to_excel(
        writer,
        sheet_name="Turnos presenciais",
        index=False
    )


# ------------------------------------------------------------
# 17. CRIAR README PARA O GITHUB
# ------------------------------------------------------------

texto_readme = f"""
# Universidade fica mais velha

Análise dos ingressantes do ensino superior brasileiro com base nos
microdados do Censo da Educação Superior de 2015, do Inep.

## Resultados gerais de 2015

- Total de ingressantes: {total_ingressantes:,}
- Ingressantes com 30 anos ou mais: {total_30_mais:,}
- Participação dos ingressantes 30+: {total_30_mais / total_ingressantes:.2%}

## Fonte

Instituto Nacional de Estudos e Pesquisas Educacionais Anísio Teixeira
(Inep), Censo da Educação Superior de 2015.

## Metodologia

A análise utiliza o arquivo `MICRODADOS_CADASTRO_CURSOS_2015.CSV`.

A base pública está agregada no nível dos cursos. O total de ingressantes
é obtido pela variável `QT_ING`.

Os ingressantes com 30 anos ou mais correspondem à soma das variáveis:

- `QT_ING_30_34`
- `QT_ING_35_39`
- `QT_ING_40_49`
- `QT_ING_50_59`
- `QT_ING_60_MAIS`

A modalidade de ensino é identificada por `TP_MODALIDADE_ENSINO`.
A rede pública ou privada é identificada por `TP_REDE`.

Pedagogia e Serviço Social foram identificados pelos códigos Cine:

- Pedagogia: `0113P01`
- Serviço Social: `0923S01`

## Limitação

A divulgação pública apresenta separadamente os ingressantes por faixa
etária e por turno. Por isso, não é possível calcular diretamente o
cruzamento entre faixa etária e turno.

## Reprodução

O arquivo original do Inep não foi incluído neste repositório. Para
reproduzir a análise, baixe os microdados de 2015 no site do Inep e
execute o notebook do projeto.
"""

texto_readme = texto_readme.replace(
    f"{total_ingressantes:,}",
    f"{total_ingressantes:,}".replace(",", ".")
).replace(
    f"{total_30_mais:,}",
    f"{total_30_mais:,}".replace(",", ".")
)

(pasta_projeto / "README.md").write_text(
    texto_readme.strip(),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 18. COMPACTAR E BAIXAR OS RESULTADOS
# ------------------------------------------------------------

caminho_pacote = shutil.make_archive(
    "/content/analise_inep_2015",
    "zip",
    root_dir=pasta_projeto
)

print("\nAnálise concluída.")
print(f"Pacote criado: {caminho_pacote}")
print("O download começará automaticamente.")

files.download(caminho_pacote)

Selecione o ZIP dos microdados de 2015.


Saving microdados_censo_da_educacao_superior_2015 (1).zip to microdados_censo_da_educacao_superior_2015 (1) (2).zip

Arquivo escolhido: microdados_censo_da_educacao_superior_2015 (1) (2).zip
CSV localizado: MICRODADOS_CADASTRO_CURSOS_2015.CSV

Linhas: 81,156
Colunas utilizadas: 21

CHECAGENS


,checagem,valor,resultado_esperado,status
0,Total de ingressantes,2922400,2922400,OK
1,Soma das faixas etárias,2922400,2922400,OK
2,Linhas em que idade não soma QT_ING,0,0,OK



INGRESSANTES POR FAIXA ETÁRIA


,faixa_etaria,ingressantes,percentual_total
0,Até 17,"33,164",1.13%
1,18 a 24,"1,619,680",55.42%
2,25 a 29,"484,934",16.59%
3,30 a 34,"333,256",11.40%
4,35 a 39,"210,215",7.19%
5,40 a 49,"182,785",6.25%
6,50 a 59,"51,195",1.75%
7,60 ou mais,"7,171",0.25%


...


TURNOS — CURSOS PRESENCIAIS


,turno,ingressantes,percentual
0,Diurno,"816,484",36.65%
1,Noturno,"1,411,357",63.35%



Análise concluída.
Pacote criado: /content/analise_inep_2015.zip
O download começará automaticamente.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>